# ctl-stop-harness: Gemini on Vertex AI batch run

Runs the LangGraph payments agent with a real planning model (Gemini on Vertex AI), 20 runs each of `conformant` and `stale_plan`, and analyses the result.

**Before you start**
- Colab Secrets (key icon, left bar): `GCP_PROJECT` (your Google Cloud project ID). Switch "Notebook access" on.
- Sign in with a Google account that has the *Vertex AI User* role on the project.
- The project ID is read from the secret and never written into this notebook, the repo or the evidence bundles.

**Order:** 1 sign in, 2 model check, 3 get the code, 4 one test run, 5 the batch (about an hour, about $1.80), 6 analysis, 7 package the evidence.

## 1. Install and sign in

In [ ]:
# Colab runs as you. authenticate_user() gives this session Application Default Credentials
# for the Google account you pick, which the Gemini client then uses.
!pip -q install --upgrade google-genai

from google.colab import auth, userdata
auth.authenticate_user()

# Project ID comes from a Colab Secret, so it is never stored in the notebook.
PROJECT = userdata.get("GCP_PROJECT")

## 2. Check the model is reachable
The model name and location must match. On 2026-11-16 `gemini-3.5-flash` answered only in the `global` location (404 in `us-central1`), and `gemini-2.5-flash` had been retired.

In [ ]:
from google import genai

MODEL = "gemini-3.5-flash"
for loc in ["us-central1", "global"]:
    client = genai.Client(vertexai=True, project=PROJECT, location=loc)
    try:
        r = client.models.generate_content(model=MODEL, contents="Reply with the single word: ready")
        print("OK  ", MODEL, loc, "->", r.text.strip())
    except Exception as e:
        # Print the error type and code only: the full message contains the project ID.
        print("fail", MODEL, loc, type(e).__name__, getattr(e, "code", ""))

## 3. Get the harness code (public repository)

In [ ]:
import os

REPO = "https://github.com/nvns10/ctl-stop-harness.git"

if not os.path.exists("/content/ctl-stop-harness"):
    !git clone -q {REPO}
%cd /content/ctl-stop-harness
!git pull -q

# Installs the harness plus its Vertex extra (langgraph, langchain-core, google-genai).
!pip -q install -e ".[vertex]"
!git log --oneline -1

## 4. One test run (conformant)
The planner reads the project and location from environment variables. Expect PASS on all three controls and a loaded stop latency near 0.0 s.

In [ ]:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT      # read by the planner, never written to evidence
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

!python -W ignore -m stopharness run --target langgraph_payments --planner vertex --mode conformant --out ev-test --no-mapping --trials 5 | grep -E "^\| CTL|loaded:|t_stop_s"

## 5. The batch

Each run is a full test of all three controls against one agent mode, so a run is one row in `stats/runs.csv`. Modes are shuffled within each pair so time of day and service load affect both equally.

- `conformant`: checks for a stop before planning and again before acting (expected PASS).
- `stale_plan`: checks only before planning, so a plan made before the stop can still be carried out after it (the defect).

The cell can be stopped and re-run: it skips run IDs already in `stats/runs.csv`. To change any setting, delete `stats/` first (`!rm -rf stats`) so old and new rows are not mixed. Estimated cost is about $0.046 per run.

In [ ]:
import json, glob, subprocess, csv, random, time, datetime, statistics

N_PER_MODE = 20      # runs per mode (40 in total)
TRIALS = 5           # trials per condition inside each run (the harness minimum is 3)
OUT = "stats"; os.makedirs(OUT, exist_ok=True)
CSV = f"{OUT}/runs.csv"

# List prices per million tokens (third-party listings, not yet confirmed on Google's page).
# Thinking tokens are billed as output. Treat est_cost_usd as an estimate; check the billing report.
PRICE_IN, PRICE_OUT = 1.50, 9.00

FIELDS = ["run_id","mode","order","started_utc","exit_code","v003","v004","v005",
          "loaded_max_stop_s","baseline_max_stop_s","t_stop_003_s","post_stop_003",
          "plan_calls","plan_lat_median_s","plan_lat_max_s","plan_discarded",
          "plan_errors","plan_retries","input_tokens","output_tokens","thought_tokens","est_cost_usd","error"]

# Fixed seed so the shuffled order can be reproduced.
rng = random.Random(20261116)
schedule = []
for i in range(N_PER_MODE):
    pair = ["conformant", "stale_plan"]; rng.shuffle(pair)
    schedule += [(f"r{i:02d}-{m}", m) for m in pair]

# Resume support: skip runs already recorded.
done = set()
if os.path.exists(CSV):
    done = {r["run_id"] for r in csv.DictReader(open(CSV))}
else:
    csv.DictWriter(open(CSV, "w", newline=""), FIELDS).writeheader()

def one_run(run_id, mode, order):
    """Run the harness once and summarise the evidence bundle as one CSV row."""
    bundle = f"{OUT}/{run_id}"
    t0 = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    p = subprocess.run(["python", "-W", "ignore", "-m", "stopharness", "run",
        "--target", "langgraph_payments", "--planner", "vertex", "--mode", mode,
        "--out", bundle, "--no-mapping", "--trials", str(TRIALS)],
        capture_output=True, text=True)
    row = dict.fromkeys(FIELDS)
    row.update(run_id=run_id, mode=mode, order=order, started_utc=t0, exit_code=p.returncode)
    try:
        # Verdicts and measurements come from the harness's own results file.
        res = json.load(open(f"{bundle}/results.json"))["outcomes"]
        by = {o["control_id"]: o for o in res}
        row["v003"], row["v004"], row["v005"] = (by[c]["result"] for c in ("CTL-STOP-003", "CTL-STOP-004", "CTL-STOP-005"))
        cond = by["CTL-STOP-004"]["metrics"].get("conditions", {})
        row["loaded_max_stop_s"] = cond.get("loaded", {}).get("max")      # longest stop delay under load
        row["baseline_max_stop_s"] = cond.get("baseline", {}).get("max")
        m3 = by["CTL-STOP-003"]["metrics"]
        row["t_stop_003_s"], row["post_stop_003"] = m3.get("t_stop_s"), m3.get("post_stop_initiations")

        # Model-call details come from the event logs: latency, discarded plans, retries, tokens.
        ev = []
        for f in glob.glob(f"{bundle}/events/*.jsonl"):
            ev += [json.loads(l) for l in open(f)]
        dec = [e for e in ev if e.get("type") == "plan_decision"]
        lat = [e["data"]["latency_s"] for e in dec if "latency_s" in e.get("data", {})]
        row["plan_calls"] = len(lat)
        if lat:
            row["plan_lat_median_s"], row["plan_lat_max_s"] = statistics.median(lat), max(lat)
        row["plan_discarded"] = sum(1 for e in ev if e.get("type") == "plan_discarded")   # plans dropped at the act check
        row["plan_errors"] = sum(1 for e in ev if e.get("type") == "plan_error")           # model calls that still failed
        row["plan_retries"] = sum(e["data"].get("retries", 0) for e in dec)                # transient errors retried
        tok = lambda k: sum(e["data"].get(k, 0) for e in dec)
        row["input_tokens"], row["output_tokens"], row["thought_tokens"] = tok("input_tokens"), tok("output_tokens"), tok("thought_tokens")
        row["est_cost_usd"] = round((row["input_tokens"] * PRICE_IN + (row["output_tokens"] + row["thought_tokens"]) * PRICE_OUT) / 1e6, 5)
    except Exception as exc:
        # Keep the row so a failed run is counted, not silently dropped.
        row["error"] = f"{type(exc).__name__}: {exc} | {p.stderr[-300:]}"
    return row

# Output per line: index, run id, verdicts for 003 / 004 / 005, longest stop delay (s), estimated cost ($).
for order, (run_id, mode) in enumerate(schedule):
    if run_id in done:
        continue
    row = one_run(run_id, mode, order)
    with open(CSV, "a", newline="") as fh:
        csv.DictWriter(fh, FIELDS).writerow(row)
    print(order, run_id, row["v003"], row["v004"], row["v005"], row["loaded_max_stop_s"], row["est_cost_usd"], row["error"] or "")
    time.sleep(5)    # brief pause between runs to stay clear of rate limits

## 6. Analysis
Uses the same script as the repository (`scripts/analyze_batch.py`): Wilson intervals, Fisher exact test, Mann-Whitney U, bootstrap intervals, Spearman correlation, data quality and estimated spend. Failures and INCONCLUSIVE results are reported, not dropped.

In [ ]:
!pip -q install pandas scipy numpy
!python scripts/analyze_batch.py stats/runs.csv

In [ ]:
# Anything other than a clean PASS, for a closer look (run id, verdicts, delay, retries).
import pandas as pd
pd.set_option("display.width", 200)
df = pd.read_csv("stats/runs.csv")
show = ["run_id","v003","v004","v005","loaded_max_stop_s","baseline_max_stop_s","plan_retries","plan_errors","error"]
df[(df[["v003","v004","v005"]] != "PASS").any(axis=1) | df["error"].notna()][show]

## 7. Package the evidence
Checks that the project ID is not in any file, then zips `stats/` and downloads it. Do not share the zip before the contract review.

In [ ]:
import shutil
from google.colab import files

# Search every file for the project ID (the command is not echoed). Only a clean result is downloaded.
r = subprocess.run(f"grep -rIl '{PROJECT}' stats/ || echo clean.", shell=True, capture_output=True, text=True)
print("grep result:", r.stdout.strip())

if r.stdout.strip() == "clean.":
    shutil.make_archive("vertex-stats-evidence", "zip", "stats")
    print("zip size (MB):", round(os.path.getsize("vertex-stats-evidence.zip") / 1e6, 2))
    files.download("vertex-stats-evidence.zip")
else:
    print("Do NOT download. A file contains the project ID; paste the list above to scrub it.")

## After the run
- Billing: the cost can take about a day to appear. In Cloud Billing, Reports, group by SKU and compare with the estimate printed in section 6.
- The earlier gcloud permission test is not needed: the Vertex AI User role is enough to call the model, and listing enabled services needs a different permission.